In [1]:
# ============================================================
# KPI DICTIONARY & DATA QUALITY CONTRACT
# RabTech Academy - Data Analytics & Business Intelligence
# ============================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime
from google.colab import files
from IPython.display import display, Markdown

print("KPI Dictionary & Data Quality Contract")
print("=" * 60)

# ------------------------------------------------------------
# 1. UPLOAD FILES
# ------------------------------------------------------------

print("\nUpload the 2 files given by RabTech Academy:")
print("1. Raw retail orders dataset")
print("2. Retail data dictionary")

uploaded = files.upload()

print("\nUploaded files:")
for f in uploaded.keys():
    print(" -", f)


# ------------------------------------------------------------
# 2. IDENTIFY DATASET AND DATA DICTIONARY
# ------------------------------------------------------------

csv_files = list(uploaded.keys())

if len(csv_files) < 2:
    raise ValueError("Please upload both CSV files.")

data_file = None
dictionary_file = None

for f in csv_files:
    temp = pd.read_csv(f)
    cols = set(temp.columns)

    if "order_id" in cols and "order_date" in cols:
        data_file = f

    if "column" in cols and "quality_rule" in cols:
        dictionary_file = f

if data_file is None:
    raise ValueError("Raw retail orders dataset was not detected.")

if dictionary_file is None:
    raise ValueError("Retail data dictionary was not detected.")


# ------------------------------------------------------------
# 3. READ FILES
# ------------------------------------------------------------

df = pd.read_csv(data_file)
dictionary = pd.read_csv(dictionary_file)

print("\nRaw Dataset:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df)

print("\nRetail Data Dictionary:")
display(dictionary)


# ------------------------------------------------------------
# 4. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

dictionary.columns = (
    dictionary.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


# ------------------------------------------------------------
# 5. HELPER FUNCTIONS
# ------------------------------------------------------------

def normalize_text(series):
    return series.astype("string").str.strip()


def normalize_category(series):
    return (
        series.astype("string")
        .str.strip()
        .str.title()
    )


# ------------------------------------------------------------
# 6. CREATE PROFILE DATA
# ------------------------------------------------------------

profile = []

for col in df.columns:

    missing = df[col].isna().sum()
    missing_pct = round((missing / len(df)) * 100, 2)

    unique = df[col].nunique(dropna=True)
    duplicate_values = len(df) - unique - missing

    profile.append({
        "column": col,
        "data_type": str(df[col].dtype),
        "rows": len(df),
        "missing_count": missing,
        "missing_pct": missing_pct,
        "unique_values": unique,
        "duplicate_value_count": duplicate_values
    })

profile_df = pd.DataFrame(profile)


# ------------------------------------------------------------
# 7. DATA QUALITY CHECKS
# ------------------------------------------------------------

print("\nRunning executable data-quality checks...")
print("=" * 60)

# Make temporary normalized columns
check = df.copy()

for col in ["order_id", "customer_segment", "city",
            "category", "payment_status"]:
    if col in check.columns:
        check[col] = normalize_text(check[col])


# ---------- ORDER ID ----------
order_id_complete = check["order_id"].notna().mean() * 100
order_id_unique = check["order_id"].dropna().is_unique


# ---------- ORDER DATE ----------
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # First try ISO YYYY-MM-DD
    try:
        return pd.to_datetime(
            value,
            format="%Y-%m-%d"
        )
    except:
        pass

    # Then try DD/MM/YYYY
    try:
        return pd.to_datetime(
            value,
            format="%d/%m/%Y"
        )
    except:
        return pd.NaT


parsed_dates = check["order_date"].apply(parse_mixed_date)

date_complete = parsed_dates.notna().mean() * 100

today = pd.Timestamp.today().normalize()

date_valid = (
    parsed_dates.notna()
    & (parsed_dates >= pd.Timestamp("2025-01-01"))
    & (parsed_dates <= today)
)

date_valid_pct = date_valid.mean() * 100


# ---------- CUSTOMER SEGMENT ----------
valid_segments = {
    "Student",
    "Fresher",
    "Professional"
}

segments = normalize_category(check["customer_segment"])

segment_valid = segments.isin(valid_segments)

segment_valid_pct = segment_valid.mean() * 100


# ---------- CITY ----------
city_complete = check["city"].notna().mean() * 100

city_nonempty = (
    check["city"].fillna("").astype(str).str.strip() != ""
)

city_valid_pct = city_nonempty.mean() * 100


# ---------- CATEGORY ----------
valid_categories = {
    "Learning Kit",
    "Course Access",
    "Mentor Session"
}

categories = normalize_category(check["category"])

category_valid = categories.isin(valid_categories)

category_valid_pct = category_valid.mean() * 100


# ---------- QUANTITY ----------
quantity_numeric = pd.to_numeric(
    check["quantity"],
    errors="coerce"
)

quantity_valid = (
    quantity_numeric.notna()
    & (quantity_numeric > 0)
    & (quantity_numeric % 1 == 0)
)

quantity_valid_pct = quantity_valid.mean() * 100


# ---------- UNIT PRICE ----------
unit_price = pd.to_numeric(
    check["unit_price"],
    errors="coerce"
)

unit_price_valid = (
    unit_price.notna()
    & (unit_price >= 0)
)

unit_price_valid_pct = unit_price_valid.mean() * 100


# ---------- DISCOUNT ----------
discount = pd.to_numeric(
    check["discount_pct"],
    errors="coerce"
)

# Missing discount is treated as a separate completeness issue.
discount_valid = (
    discount.isna()
    | ((discount >= 0) & (discount <= 100))
)

discount_valid_pct = discount_valid.mean() * 100
discount_missing_pct = discount.isna().mean() * 100


# ---------- PAYMENT STATUS ----------
valid_payment_status = {
    "Paid",
    "Pending",
    "Failed",
    "Refunded"
}

payment_status = normalize_category(
    check["payment_status"]
)

payment_valid = payment_status.isin(
    valid_payment_status
)

payment_valid_pct = payment_valid.mean() * 100


# ------------------------------------------------------------
# 8. DUPLICATE CHECK
# ------------------------------------------------------------

duplicate_order_rows = check["order_id"].duplicated(
    keep=False
).sum()

duplicate_order_pct = (
    duplicate_order_rows / len(check)
) * 100


# ------------------------------------------------------------
# 9. FRESHNESS CHECK
# ------------------------------------------------------------

# The supplied dataset does not contain an ingestion timestamp.
# Therefore true pipeline freshness cannot be measured from
# the source file alone.

freshness_status = "NOT MEASURABLE"
freshness_note = (
    "Source file does not contain an ingestion/updated timestamp. "
    "Contract requires ingestion timestamp for production freshness."
)


# ------------------------------------------------------------
# 10. DATA QUALITY RESULT TABLE
# ------------------------------------------------------------

quality_results = pd.DataFrame({

    "Dimension": [
        "Completeness",
        "Uniqueness",
        "Validity",
        "Consistency",
        "Freshness"
    ],

    "Metric": [
        "Required fields populated",
        "Order ID uniqueness",
        "Values follow business rules",
        "Standard categories/status values",
        "Source ingestion timestamp"
    ],

    "Result": [
        round(
            check.notna().mean().mean() * 100,
            2
        ),
        round(
            (1 - duplicate_order_pct / 100) * 100,
            2
        ),
        round(
            np.mean([
                date_valid_pct,
                segment_valid_pct,
                city_valid_pct,
                category_valid_pct,
                quantity_valid_pct,
                unit_price_valid_pct,
                discount_valid_pct,
                payment_valid_pct
            ]),
            2
        ),
        round(
            np.mean([
                segment_valid_pct,
                category_valid_pct,
                payment_valid_pct
            ]),
            2
        ),
        freshness_status
    ],

    "Threshold": [
        ">= 95%",
        "100%",
        ">= 98%",
        ">= 98%",
        "<= 24 hours"
    ]
})


# ------------------------------------------------------------
# 11. DISPLAY QUALITY RESULTS
# ------------------------------------------------------------

print("\nDATA QUALITY SUMMARY")
print("=" * 60)

display(quality_results)


# ------------------------------------------------------------
# 12. DETAILED CHECK TABLE
# ------------------------------------------------------------

detailed_checks = pd.DataFrame({

    "Check": [
        "Order ID completeness",
        "Order ID uniqueness",
        "Order date completeness",
        "Order date validity",
        "Customer segment validity",
        "City completeness",
        "Category validity",
        "Quantity validity",
        "Unit price validity",
        "Discount validity",
        "Discount completeness",
        "Payment status validity",
        "Freshness"
    ],

    "Result": [
        f"{order_id_complete:.2f}%",
        "PASS" if order_id_unique else "FAIL",
        f"{date_complete:.2f}%",
        f"{date_valid_pct:.2f}%",
        f"{segment_valid_pct:.2f}%",
        f"{city_complete:.2f}%",
        f"{category_valid_pct:.2f}%",
        f"{quantity_valid_pct:.2f}%",
        f"{unit_price_valid_pct:.2f}%",
        f"{discount_valid_pct:.2f}%",
        f"{100-discount_missing_pct:.2f}%",
        f"{payment_valid_pct:.2f}%",
        freshness_status
    ],

    "Threshold": [
        "100%",
        "100%",
        "100%",
        "100%",
        ">= 98%",
        "100%",
        ">= 98%",
        "100%",
        "100%",
        "100%",
        ">= 95%",
        ">= 98%",
        "<= 24 hours"
    ]
})

print("\nDETAILED DATA QUALITY CHECKS")
display(detailed_checks)


# ------------------------------------------------------------
# 13. CREATE CLEANED DATASET
# ------------------------------------------------------------

clean = df.copy()

# Text normalization
clean["order_id"] = normalize_text(clean["order_id"])
clean["customer_segment"] = normalize_category(
    clean["customer_segment"]
)
clean["city"] = normalize_text(clean["city"])
clean["category"] = normalize_category(
    clean["category"]
)
clean["payment_status"] = normalize_category(
    clean["payment_status"]
)

# Date normalization
clean["order_date"] = clean["order_date"].apply(
    parse_mixed_date
)

# Numeric conversion
clean["quantity"] = pd.to_numeric(
    clean["quantity"],
    errors="coerce"
)

clean["unit_price"] = pd.to_numeric(
    clean["unit_price"],
    errors="coerce"
)

clean["discount_pct"] = pd.to_numeric(
    clean["discount_pct"],
    errors="coerce"
)

# Missing discount is assumed to be 0 only because
# the assignment dictionary explicitly permits this
# after justification.
clean["discount_pct"] = clean["discount_pct"].fillna(0)


# Remove clearly invalid rows
clean = clean[
    clean["order_id"].notna()
]

clean = clean[
    clean["order_date"].notna()
]

clean = clean[
    clean["quantity"].notna()
]

clean = clean[
    clean["quantity"] > 0
]

clean = clean[
    clean["quantity"] % 1 == 0
]

clean = clean[
    clean["unit_price"].notna()
    & (clean["unit_price"] >= 0)
]

clean = clean[
    clean["discount_pct"].between(0, 100)
]

clean = clean[
    clean["customer_segment"].isin(valid_segments)
]

clean = clean[
    clean["category"].isin(valid_categories)
]

clean = clean[
    clean["payment_status"].isin(valid_payment_status)
]

# Remove duplicate order IDs
clean = clean.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

clean = clean.reset_index(drop=True)


# ------------------------------------------------------------
# 14. KPI CALCULATIONS
# ------------------------------------------------------------

clean["gross_sales"] = (
    clean["quantity"]
    * clean["unit_price"]
)

clean["discount_amount"] = (
    clean["gross_sales"]
    * clean["discount_pct"]
    / 100
)

clean["net_sales"] = (
    clean["gross_sales"]
    - clean["discount_amount"]
)


# KPI values
total_orders = clean["order_id"].nunique()

total_units = clean["quantity"].sum()

gross_sales = clean["gross_sales"].sum()

discount_amount = clean["discount_amount"].sum()

net_sales = clean["net_sales"].sum()

aov = (
    net_sales / total_orders
    if total_orders > 0
    else 0
)

paid_order_rate = (
    (clean["payment_status"] == "Paid").mean()
    * 100
)

refund_rate = (
    (clean["payment_status"] == "Refunded").mean()
    * 100
)


# ------------------------------------------------------------
# 15. KPI DICTIONARY
# ------------------------------------------------------------

kpi_dictionary = pd.DataFrame({

    "KPI": [
        "Total Orders",
        "Total Units Sold",
        "Gross Sales",
        "Discount Amount",
        "Net Sales",
        "Average Order Value (AOV)",
        "Paid Order Rate",
        "Refund Rate"
    ],

    "Formula": [
        "COUNT(DISTINCT order_id)",
        "SUM(quantity)",
        "SUM(quantity × unit_price)",
        "SUM(gross_sales × discount_pct / 100)",
        "SUM(gross_sales - discount_amount)",
        "Net Sales / Total Orders",
        "Paid Orders / Total Orders × 100",
        "Refunded Orders / Total Orders × 100"
    ],

    "Grain": [
        "Order",
        "Order line",
        "Order line",
        "Order line",
        "Order line",
        "Order",
        "Order",
        "Order"
    ],

    "Filters": [
        "Valid order_id",
        "Valid positive quantity",
        "Valid quantity and unit price",
        "Valid discount percentage",
        "Valid order records",
        "Valid order records",
        "payment_status = Paid",
        "payment_status = Refunded"
    ],

    "Owner": [
        "Retail Operations Manager",
        "Retail Operations Manager",
        "Finance Manager",
        "Finance Manager",
        "Finance Manager",
        "Business Intelligence Manager",
        "Finance Manager",
        "Finance Manager"
    ],

    "Refresh Cadence": [
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily",
        "Daily"
    ]
})


# ------------------------------------------------------------
# 16. KPI VALUES
# ------------------------------------------------------------

kpi_values = pd.DataFrame({

    "KPI": [
        "Total Orders",
        "Total Units Sold",
        "Gross Sales",
        "Discount Amount",
        "Net Sales",
        "Average Order Value (AOV)",
        "Paid Order Rate",
        "Refund Rate"
    ],

    "Value": [
        total_orders,
        total_units,
        round(gross_sales, 2),
        round(discount_amount, 2),
        round(net_sales, 2),
        round(aov, 2),
        round(paid_order_rate, 2),
        round(refund_rate, 2)
    ]
})


# ------------------------------------------------------------
# 17. DATA QUALITY CONTRACT
# ------------------------------------------------------------

contract = f"""
# Data Quality Contract

## Dataset
Raw Retail Orders Dataset

## Business Purpose
The dataset is used by the Retail Operations, Finance,
and Business Intelligence teams to monitor sales,
orders, discounts, payments, and customer segments.

## Decision Owner
Retail Operations Manager

## Data Provider
Retail Data / Operations Team

## Refresh Cadence
Daily

## Quality Dimensions

### 1. Completeness
Required fields must be populated.

Critical fields:
- order_id
- order_date
- city
- quantity
- unit_price

Threshold:
- Critical fields: 100%
- Other required fields: >= 95%

Action:
Records below threshold are rejected or sent for correction.

### 2. Uniqueness
order_id must be unique.

Threshold:
100% unique.

Action:
Duplicate orders are quarantined and investigated by
the Retail Operations Manager.

### 3. Validity
Values must follow the business rules defined in the
Retail Data Dictionary.

Examples:
- quantity must be a whole number greater than zero
- unit_price must be non-negative
- discount_pct must be between 0 and 100
- order_date must be valid
- category must be a valid product category

Threshold:
>= 98% overall and 100% for critical financial fields.

Action:
Invalid records are quarantined before KPI reporting.

### 4. Consistency
Categorical fields must use standardized values.

Customer segment:
- Student
- Fresher
- Professional

Category:
- Learning Kit
- Course Access
- Mentor Session

Payment status:
- Paid
- Pending
- Failed
- Refunded

Threshold:
>= 98%

Action:
Normalize casing and whitespace automatically.
Unrecognized values are rejected for investigation.

### 5. Freshness
The production source must contain an ingestion or
last-updated timestamp.

Threshold:
Data must be refreshed within 24 hours.

Action:
If freshness exceeds 24 hours, the BI team is alerted
and the dashboard is marked as stale.

## Escalation

Level 1:
Data analyst investigates failed records.

Level 2:
Retail Operations Manager is notified for business-rule
or source-data problems.

Level 3:
Finance / BI Manager is notified when financial KPIs
are affected.

## Current Dataset Findings

Rows received: {len(df)}

Duplicate order rows: {duplicate_order_rows}

Invalid date rows: {(~date_valid).sum()}

Missing city rows: {check["city"].isna().sum()}

Invalid quantity rows: {(~quantity_valid).sum()}

Invalid discount rows: {(~discount_valid).sum()}

Freshness:
{freshness_status}

Note:
{freshness_note}
"""


# ------------------------------------------------------------
# 18. SAVE ALL DELIVERABLES
# ------------------------------------------------------------

# Excel workbook
excel_file = "KPI_Dictionary_and_Data_Quality_Profile.xlsx"

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    kpi_dictionary.to_excel(
        writer,
        sheet_name="KPI Dictionary",
        index=False
    )

    kpi_values.to_excel(
        writer,
        sheet_name="KPI Values",
        index=False
    )

    profile_df.to_excel(
        writer,
        sheet_name="Data Profile",
        index=False
    )

    quality_results.to_excel(
        writer,
        sheet_name="Quality Summary",
        index=False
    )

    detailed_checks.to_excel(
        writer,
        sheet_name="Detailed Checks",
        index=False
    )

    dictionary.to_excel(
        writer,
        sheet_name="Original Dictionary",
        index=False
    )


# Clean dataset
clean.to_csv(
    "cleaned_retail_orders.csv",
    index=False
)


# Quality contract
with open(
    "Data_Quality_Contract.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(contract)


# ------------------------------------------------------------
# 19. DISPLAY FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("ASSIGNMENT COMPLETED")
print("=" * 60)

print("\nKPI VALUES:")
display(kpi_values)

print("\nCLEANED DATASET:")
print("Rows before cleaning :", len(df))
print("Rows after cleaning  :", len(clean))

display(clean.head())


print("\nFILES CREATED:")
print("1. KPI_Dictionary_and_Data_Quality_Profile.xlsx")
print("2. cleaned_retail_orders.csv")
print("3. Data_Quality_Contract.md")


# ------------------------------------------------------------
# 20. DOWNLOAD FILES
# ------------------------------------------------------------

files.download(
    "KPI_Dictionary_and_Data_Quality_Profile.xlsx"
)

files.download(
    "cleaned_retail_orders.csv"
)

files.download(
    "Data_Quality_Contract.md"
)

print("\nAll deliverables have been generated successfully.")

KPI Dictionary & Data Quality Contract

Upload the 2 files given by RabTech Academy:
1. Raw retail orders dataset
2. Retail data dictionary


Saving retail-data-dictionary.csv to retail-data-dictionary.csv
Saving retail-orders-raw.csv to retail-orders-raw.csv

Uploaded files:
 - retail-data-dictionary.csv
 - retail-orders-raw.csv

Raw Dataset:
Rows: 12
Columns: 9


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid



Retail Data Dictionary:


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All deliverables have been generated successfully.
